# Kaggle image edit detection training
Enable GPU and Internet. Attach a dataset containing images, masks and a JSON manifest.
Each record needs image, mask (edited only), label, source_id and split (train/val/test).
Paths may be absolute Kaggle paths or relative to DATA_ROOT. Legacy project paths are remapped only when the attached data preserves the project directory layout.
The code does not download or invent 10,000 examples. Counts are printed before training.
Upload the updated repository as a Kaggle dataset, or push the local fixes to GitHub before using the clone option. A local edit does not update GitHub.
The 9.5-hour training budget leaves time for evaluation, but validation/evaluation duration depends on data size; this is not a guaranteed 12-hour finish.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
# EDIT THESE THREE PATHS to match attached Kaggle datasets.
CODE_SOURCE = Path('/kaggle/input/kaggle1')
DATA_ROOT = Path('/kaggle/input/ffdm-data')
MANIFEST = DATA_ROOT / 'manifest.json'
REPO = Path('/kaggle/working/ffdm')
OUTPUT = Path('/kaggle/working/output')
USE_GITHUB = False  # Requires the updated files to have been pushed first.
GIT_REF = None      # Set an exact commit for reproducibility when cloning.
RESUME = None      # latest_resume.pt; restore best_superframe_kaggle.pt to OUTPUT too.
INITIALIZE = None  # Optional explicitly attached compatible initial checkpoint.
def run(args):
    subprocess.run([str(a) for a in args], check=True, env=os.environ.copy())
run(['nvidia-smi'])
if not REPO.exists():
    if USE_GITHUB:
        if not GIT_REF: raise ValueError('Set GIT_REF to the commit containing the fixes')
        run(['git','clone','https://github.com/Dakshankarthic/ffdm.git',REPO])
        subprocess.run(['git','checkout',GIT_REF],cwd=REPO,check=True)
    else:
        candidates = [CODE_SOURCE] if CODE_SOURCE.is_dir() else []
        candidates += sorted(p for p in Path('/kaggle/input').rglob('kaggle1') if p.is_dir())
        marker = Path('controlled_localization/src/controlled_localization/kaggle_prepare.py')
        roots = [p for base in candidates for p in (base, base/'ffdm') if (p/marker).is_file()]
        archives = [p for base in candidates for p in base.rglob('kaggle_ffdm_code.zip')]
        if roots:
            shutil.copytree(roots[0], REPO)
        elif archives:
            shutil.unpack_archive(archives[0], REPO.parent)
        else:
            raise FileNotFoundError('Attach the kaggle1 dataset containing kaggle_ffdm_code.zip or the ffdm folder.')
os.chdir(REPO)
os.environ['PYTHONPATH'] = str(REPO/'controlled_localization/src')
sys.path.insert(0,os.environ['PYTHONPATH'])
for name in ['kaggle_prepare.py','kaggle_evaluate.py','train_kaggle_dual_t4.py']:
    if not (Path(os.environ['PYTHONPATH'])/'controlled_localization'/name).is_file():
        raise RuntimeError('Outdated code attachment: missing '+name)
run([sys.executable,'-m','pip','install','-q','timm','pyarrow','scikit-learn','huggingface_hub','scipy','Pillow'])
OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator'
print('GPUs:',torch.cuda.device_count(),'PyTorch:',torch.__version__)
if not MANIFEST.is_file():
    input_root = Path('/kaggle/input')
    bundles = sorted(input_root.rglob('kaggle_ffdm_data.zip'))
    if len(bundles) == 1:
        import zipfile
        DATA_ROOT = Path('/kaggle/working/ffdm-data')
        with zipfile.ZipFile(bundles[0]) as archive:
            if any(not (DATA_ROOT / member).resolve().is_relative_to(DATA_ROOT.resolve()) for member in archive.namelist()):
                raise ValueError('Invalid path in dataset archive')
            archive.extractall(DATA_ROOT)
        MANIFEST = DATA_ROOT / 'manifest.json'
    manifests = [MANIFEST] if MANIFEST.is_file() else []
    for candidate in ([] if manifests else sorted(input_root.rglob('*manifest*.json'))):
        try:
            rows = json.loads(candidate.read_text(encoding='utf-8-sig'))
        except (OSError, ValueError):
            continue
        if isinstance(rows, list) and rows and all(isinstance(r, dict) and r.get('source_id') and (r.get('image') or r.get('image_path')) for r in rows):
            if {r.get('split') for r in rows} >= {'train', 'val', 'test'}:
                manifests.append(candidate)
    if len(manifests) != 1:
        attached = ', '.join(str(p.relative_to(input_root)) for p in sorted(input_root.glob('*'))) or '(none)'
        found = ', '.join(str(p) for p in manifests) or '(none)'
        raise ValueError(f'Attach images and masks with a JSON manifest containing train, val and test records. Inputs: {attached}. Matching manifests: {found}. If several match, set MANIFEST explicitly.')
    MANIFEST = manifests[0]
    if not DATA_ROOT.is_dir(): DATA_ROOT = MANIFEST.parent
print('Training manifest:', MANIFEST)
PREPARED = OUTPUT/'manifest_kaggle.json'
run([sys.executable,'-m','controlled_localization.kaggle_prepare','--manifest',MANIFEST,'--data-root',DATA_ROOT,'--output',PREPARED])
run([sys.executable,'-m','pip','freeze'])


In [ ]:
# Global batch 4 means 2 examples per GPU with two GPUs, not 4 per GPU.
# First use STEPS=2 and VAL_INTERVAL=1 for a smoke run in a separate OUTPUT directory.
STEPS = 15000
VAL_INTERVAL = 500
BATCH_SIZE = 4
MAX_HOURS = 9.5
if not RESUME and (OUTPUT/'best_superframe_kaggle.pt').exists():
    raise RuntimeError('Existing run: set RESUME or choose a new output directory')
args=[sys.executable,'-m','controlled_localization.train_kaggle_dual_t4',
      '--manifest',PREPARED,'--output-dir',OUTPUT,'--steps',STEPS,
      '--batch_size',BATCH_SIZE,'--val_interval',VAL_INTERVAL,'--max-hours',MAX_HOURS]
if RESUME: args+=['--resume',RESUME]
if INITIALIZE: args+=['--initialize',INITIALIZE]
run(args)


In [ ]:
# Evaluates the NEW checkpoint, calibrating only on validation.
AUDIT = OUTPUT/'kaggle_evaluation.json'
if AUDIT.exists(): AUDIT.unlink()  # Prevent displaying a stale report after failure.
run([sys.executable,'-m','controlled_localization.kaggle_evaluate',
     '--manifest',PREPARED,'--checkpoint',OUTPUT/'best_superframe_kaggle.pt','--output',AUDIT])
result=json.loads(AUDIT.read_text())
for name,res in result.items():
    print(name,res['checkpoint_sha256'])
    for score,metrics in res['frozen_test_results'].items():
        print(score,metrics['accuracy'],metrics['balanced_accuracy'],metrics['auroc'],metrics['confusion'])


# Can also be run independently after interruption to preserve checkpoints.
required=[OUTPUT/'best_superframe_kaggle.pt',OUTPUT/'manifest_kaggle.json']
for p in required:
    if not p.is_file(): raise FileNotFoundError(p)
archive=shutil.make_archive('/kaggle/working/kaggle_training_artifacts','zip',OUTPUT)
print('Saved:',archive)
print('Evaluation included:',(OUTPUT/'kaggle_evaluation.json').exists())
